In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import json
from utils.preprocess import normalize_text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import pickle

In [3]:
with open("../data/intents.json", "r") as file:
    data = json.load(file)

patterns = []
labels = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        patterns.append(pattern)
        labels.append(intent["tag"])

print("Total samples:", len(patterns))
print(patterns[:3])
print(labels[:3])

Total samples: 229
["I am so unhappy, why can't I just do things right?", 'I am feeling very low functioning right now.', 'Nothing makes me happy anymore.']
['sadness', 'sadness', 'sadness']


In [4]:
clean_patterns = [normalize_text(p) for p in patterns]

print(clean_patterns[:5])

['unhappy cant thing right', 'feeling low functioning right', 'nothing make happy anymore', 'selfish cry public feel awful', 'cry cant even determine im happy sad']


In [5]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(clean_patterns)

print("Shape:", X.shape)

Shape: (229, 693)


In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

In [7]:
NLP_model = MultinomialNB()
NLP_model.fit(X, y)

print("Model trained!")

Model trained!


In [8]:
def predict_intent(text):
    clean = normalize_text(text)
    vec = vectorizer.transform([clean])
    
    pred = NLP_model.predict(vec)[0]
    intent = label_encoder.inverse_transform([pred])[0]
    
    probs = NLP_model.predict_proba(vec)[0]
    confidence = max(probs)
    
    return intent, float(confidence)

In [9]:
print(predict_intent("I feel very anxious today"))
print(predict_intent("I am not okay"))
print(predict_intent("thanks a lot"))
print(predict_intent("hello"))

(np.str_('anxiety'), 0.20140133400228744)
(np.str_('sadness'), 0.22779192922303404)
(np.str_('gratitude'), 0.2567681117028672)
(np.str_('greeting'), 0.2213740699056158)


In [10]:
with open("../models/NLP_model.pkl", "wb") as f:
    pickle.dump(NLP_model, f)

with open("../models/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Saved all files")

Saved all files


In [11]:
import pickle
import json
from utils.preprocess import normalize_text
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [12]:
with open("../models/NLP_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("../models/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("../models/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

with open("../data/intents.json") as f:
    intents = json.load(f)

In [13]:
intent_responses = {}

for intent in intents["intents"]:
    intent_responses[intent["tag"]] = intent["responses"]

In [14]:
all_patterns = []
pattern_intents = []

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        all_patterns.append(pattern)
        pattern_intents.append(intent["tag"])

clean_patterns = [normalize_text(p) for p in all_patterns]
pattern_vectors = vectorizer.transform(clean_patterns)

In [15]:
crisis_keywords = [
    "kill myself", "suicide", "want to die",
    "end my life", "hurt myself", "die"
]
def is_crisis(text):
    text = text.lower()
    return any(word in text for word in crisis_keywords)

In [16]:
import random

def chatbot_response(user_input):
    
    if is_crisis(user_input):
        return random.choice(intent_responses["crisis"]), "crisis", 1.0
    
    clean = normalize_text(user_input)
    vec = vectorizer.transform([clean])
    
    pred = model.predict(vec)[0]
    intent = label_encoder.inverse_transform([pred])[0]
    
    probs = model.predict_proba(vec)[0]
    confidence = max(probs)
    
    if confidence > 0.7:
        return random.choice(intent_responses[intent]), intent, confidence
    
    else:
        similarities = cosine_similarity(vec, pattern_vectors)
        top_indices = similarities[0].argsort()[-3:][::-1]

        for idx in top_indices:
            if similarities[0][idx] > 0.3:
                fallback_intent = pattern_intents[idx]
                return random.choice(intent_responses[fallback_intent]), fallback_intent, similarities[0][idx]
        
            else:
                return "Can you tell me a bit more about how you're feeling?", "fallback", confidence
    
    
    

In [17]:
print(chatbot_response("I feel very anxious today"))
print(chatbot_response("I feel low and empty"))
print(chatbot_response("thanks a lot"))
print(chatbot_response("hello"))
print(chatbot_response("I want to die"))

("I'm here with you. Do you want to talk about what's making you feel this way?", 'anxiety', np.float64(0.6567037821576305))
('I hear you. Sometimes just expressing it can help a little.', 'sadness', np.float64(1.0))
("Happy to help. You're doing great.", 'gratitude', np.float64(1.0))
("Hi there, I'm here to listen.", 'greeting', np.float64(1.0))
("I'm really glad you shared this. It might help to talk to a close friend, family member, or a professional.", 'crisis', 1.0)
